# 🤖 Chatbot using Hugging Face Transformers
**Assignment NLP – 3 | Data Science Internship – February 2026**

**Objective:** Build a conversational chatbot using a pre-trained transformer model (DialoGPT) from Hugging Face that interacts with users and generates meaningful responses.

---
**Pipeline:** User Input → Model Processing → Response Generation → Display → Loop Until Exit

## Step 1: Install Required Libraries
Install the Hugging Face `transformers` library and `torch` (PyTorch backend).

In [ ]:
# Install dependencies (run this once in Colab or local environment)
!pip install transformers torch --quiet

## Step 2: Import Libraries

In [ ]:
# Import PyTorch for tensor operations
import torch

# AutoTokenizer: converts text to token IDs the model understands
# AutoModelForCausalLM: loads a causal language model for text generation
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 3: Load the Pre-trained Model

We use **DialoGPT-medium** by Microsoft — a GPT-2 based model specifically fine-tuned on conversational Reddit data.

| Model | Parameters | Best For |
|---|---|---|
| `microsoft/DialoGPT-small` | 117M | Fast, less accurate |
| `microsoft/DialoGPT-medium` | 345M | Balanced ✅ |
| `microsoft/DialoGPT-large` | 762M | Best quality, slower |

In [ ]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading tokenizer from '{MODEL_NAME}'...")
# Tokenizer: breaks text into tokens (sub-words) and encodes them as integers
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"⏳ Loading model from '{MODEL_NAME}'...")
# Load the causal language model (auto-selects GPU if available, else CPU)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Use GPU if available for faster inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"\n✅ Model loaded successfully on: {device}")
print(f"   Model: {MODEL_NAME}")

## Step 4: Define the Response Generation Function

This function handles:
- Encoding the conversation history + new user input
- Generating a response using the model
- Decoding the output tokens back to text

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    """
    Generate a chatbot response for a given user input.

    Args:
        user_input (str): The message typed by the user.
        chat_history_ids (torch.Tensor or None): Encoded conversation history
                          for multi-turn context. None for the first turn.

    Returns:
        response_text (str): The chatbot's reply.
        chat_history_ids (torch.Tensor): Updated conversation history.
    """

    # Step 4a: Encode user input and append End-Of-Sequence token
    # EOS token tells the model "this is where user input ends"
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"   # return PyTorch tensors
    ).to(device)

    # Step 4b: Concatenate new input with previous conversation history
    # This gives the model context from earlier turns in the conversation
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn — no previous history
        bot_input_ids = new_input_ids

    # Step 4c: Limit context to last 1000 tokens to avoid exceeding model limits
    # DialoGPT has a max context window of 1024 tokens
    if bot_input_ids.shape[-1] > 1000:
        bot_input_ids = bot_input_ids[:, -1000:]

    # Step 4d: Generate a response using the model
    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=150,          # Maximum tokens in the response
        do_sample=True,              # Enable sampling for varied responses
        top_p=0.92,                  # Nucleus sampling: consider top 92% probability mass
        top_k=50,                    # Also limit to top 50 candidate tokens
        temperature=0.75,            # Lower = more focused, Higher = more creative
        repetition_penalty=1.3,      # Penalize repeating the same phrases
        pad_token_id=tokenizer.eos_token_id  # Required to avoid warning
    )

    # Step 4e: Decode only the new response (exclude the input from output)
    # chat_history_ids[:, bot_input_ids.shape[-1]:] = tokens AFTER the input
    response_text = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True     # Remove EOS and padding tokens from output
    )

    return response_text, chat_history_ids


print("✅ Response generation function defined.")

## Step 5: Run the Chatbot (Interactive Loop)

The chatbot will:
1. Greet the user
2. Accept input from the console
3. Generate and display a response
4. Maintain conversation context across turns
5. Stop when user types `exit` or `quit`

In [ ]:
def run_chatbot():
    """
    Main chatbot loop.
    Maintains conversation history and handles user interaction.
    """

    # Initialize conversation history as None (empty at the start)
    chat_history_ids = None
    turn_count = 0  # Track number of conversation turns

    print("=" * 60)
    print(" 🤖 AI Chatbot | Powered by DialoGPT (Hugging Face)")
    print("=" * 60)
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?")
    print("\n(Type 'exit' or 'quit' to end the conversation)\n")
    print("-" * 60)

    # ── Continuous Conversation Loop ──────────────────────────────
    while True:

        # Step 5a: Accept user input
        user_input = input("You: ").strip()

        # Step 5b: Handle empty input gracefully
        if not user_input:
            print("Chatbot: It seems you didn't type anything. Please say something!\n")
            continue

        # Step 5c: Exit condition — stop when user types exit or quit
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Thank you for chatting with me. Goodbye! 👋")
            print(f"\n📊 Session summary: {turn_count} conversation turn(s) completed.")
            print("=" * 60)
            break

        # Step 5d: Generate a response
        try:
            response, chat_history_ids = generate_response(user_input, chat_history_ids)
            turn_count += 1

            # Step 5e: Display the chatbot response
            print(f"Chatbot: {response}\n")

        except Exception as e:
            # Handle unexpected errors without crashing the chatbot
            print(f"Chatbot: Sorry, I encountered an error. Let's continue! (Error: {e})\n")
            # Reset history on error to avoid corrupted state
            chat_history_ids = None


# ── Start the chatbot ──────────────────────────────────────────────
run_chatbot()

## Step 6: Sample Chatbot Output (for demonstration)

Below is a simulated demonstration of what the chatbot interaction looks like. Run the cell above for the real interactive version.

In [ ]:
# Demonstration: run a few turns automatically (no manual input needed)
# Useful to showcase chatbot behavior in notebook output

demo_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is deep learning?",
    "Thank you"
]

print("=" * 60)
print(" 🤖 DEMO: Chatbot Interaction")
print("=" * 60)
print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

chat_history_ids = None  # Reset history for demo

for user_msg in demo_inputs:
    print(f"You: {user_msg}")

    # Generate response
    response, chat_history_ids = generate_response(user_msg, chat_history_ids)
    print(f"Chatbot: {response}\n")

print("You: exit")
print("Chatbot: Thank you for chatting with me. Goodbye! 👋")
print("=" * 60)

## Summary

| Component | Details |
|---|---|
| **Model** | `microsoft/DialoGPT-medium` |
| **Framework** | PyTorch + Hugging Face Transformers |
| **Input** | Console text from the user |
| **Output** | Contextual conversational reply |
| **Context** | Multi-turn via `chat_history_ids` |
| **Exit** | Type `exit` or `quit` |

### Key concepts used:
- **Tokenization** — Text is converted to integer token IDs using `AutoTokenizer`
- **Causal Language Modeling** — Model predicts the next token given all previous tokens
- **Nucleus (top-p) Sampling** — Picks responses from the top 92% probability mass for diversity
- **Temperature** — Controls randomness of generation (lower = more deterministic)
- **Chat History** — Previous turns are concatenated so the model has conversational context
- **Repetition Penalty** — Discourages the model from repeating itself